In [ ]:
''' Phase 1B: 20 individually fishes moving in continuous space.
Objective: create 20 inidividual Golden Shiners that move through a Continuous Space'''

import numpy as np
import mesa
from mesa.experimental.continuous_space import ContinuousSpaceAgent, ContinuousSpace
from mesa.visualization import SolaraViz, make_space_component
from matplotlib.markers import MarkerStyle

class GoldenShiners(ContinuousSpaceAgent):
    """ Creatin a Golden Shienr """
    def __init__(self, model, space, position=(0, 0), speed=1.0):
        super().__init__(space, model)
        self.position = np.array(position, dtype=float)
        self.speed = speed
     
        # Random initial direction:
        angle = self.model.rng.uniform(0, 2 * np.pi)
        self.direction = np.array([np.cos(angle), np.sin(angle)])
    
    def move(self):
        """Move forward in current direction."""
        new_position = self.position + self.direction * self.speed
        self.bounce(new_position)
        self.position = new_position

    #Making sure that the agent changes it's direction when it encounters a wall:
    def bounce(self, position):
        for axis,(minimum, maximum) in enumerate(self.model.bounds):

            if position[axis] < minimum:
                position[axis] = 2 * minimum - position[axis]
                self.direction[axis] *= -1

            elif position[axis] > maximum:
                position[axis] = 2 * maximum - position[axis]
                self.direction[axis] *= -1
                

In [8]:
class GoldenShinersModel(mesa.Model):
    """Phase 1B: 20 fish."""

    def __init__(self, width=100, height=100, speed=1, n_fish= 20, seed=None):
        super().__init__(seed=seed)

        self.bounds = np.array([[0, width],[0, height] ])

        # Fixed Couzin parameters
        self.repulsion_radius = 0.5
        self.orientation_radius = 3.0
        self.attraction_radius = 5.5

        # Create continuous space
        self.space = ContinuousSpace([[0, width], [0, height]], torus=False, random=self.random)

        for _ in range(n_fish):
            # Create n fish
            position = self.rng.random(2) * np.array([width, height]) #chooses a position
            GoldenShiners(self, self.space, position,speed) #creates a fish in that position, in this case 20 fishes

    def step(self):
        """Run one simulation step."""
        self.agents.do("move")

In [ ]:
# Visualization
def agent_draw(agent):
    """Simple agent portrayal with arrow pointing in movement direction."""
    # Calculate angle from direction vector
    angle_rad = np.arctan2(agent.direction[1], agent.direction[0])
    angle_deg = np.degrees(angle_rad)
    
    # Create arrow marker pointing in agent's direction
    marker = MarkerStyle(marker='>')  # Arrow marker
    marker._transform = marker.get_transform().rotate_deg(angle_deg)
    return {"color": "yellow", "size": 15, "marker": marker}

# Set up visualization
model = GoldenShinersModel()
page = SolaraViz(
    model,
    components=[make_space_component(agent_portrayal=agent_draw)],
    name="Phase 1B: 20 Fishes"
)

page

Cannot show ipywidgets in text